# Modelamiento de la evolución molecular y la estimación de tiempos de divergencia mediante cadenas de Markov finitas.

> ***Autores:** Cristian J. Pavez Hormazábal, Camilo & Sofía*

> **Módulo Cadenas de Markov y sus Aplicaciones.**

## 1. Introducción.

La idea de que las macromoléculas biológicas registran el paso del tiempo evolutivo se remonta a Zuckerkandl y Pauling (1965), quienes, al comparar proteínas homólogas entre especies —en particular, cadenas de hemoglobina—, observaron que el número de diferencias de aminoácidos crece de forma aproximadamente lineal con el tiempo transcurrido desde la divergencia indicado por el registro fósil. Esta regularidad, conocida como **reloj molecular**, recibió poco después un fundamento teórico en la teoría neutral de Kimura (1968): si la mayoría de las sustituciones que se fijan en una población son selectivamente neutras, la tasa de sustitución por sitio coincide con la tasa de mutación y es, por tanto, aproximadamente constante en el tiempo. Desde entonces, el reloj molecular es la herramienta estándar para construir escalas temporales de la evolución, desde la datación de la separación entre dos especies hasta la calibración de árboles filogenéticos completos (Kumar, 2005).

Para traducir las diferencias observadas entre secuencias en una estimación de tiempo se necesita un modelo probabilístico del proceso de sustitución. El enfoque clásico describe la base presente en un sitio fijo del ADN, a lo largo de un linaje, como una **cadena de Markov** con espacio de estados $S=\{A,G,C,T\}$, bajo tres supuestos: (i) *propiedad de Markov*: la probabilidad de que el sitio cambie depende sólo de la base actual y no de su historia; (ii) *homogeneidad temporal*: las probabilidades de transición no varían en el tiempo, lo que constituye la formulación probabilística del reloj molecular; y (iii) los sitios evolucionan de forma *independiente e idénticamente distribuida*, de modo que un alineamiento de $L$ sitios proporciona $L$ realizaciones de la misma cadena (Yang, 2014). Si $P$ es la matriz de transición asociada a un paso de duración fija $\Delta t$, las ecuaciones de Chapman–Kolmogorov dan la probabilidad de pasar de la base $i$ a la base $j$ en $n$ pasos como $(P^n)_{ij}$ (Häggström, 2002). La literatura filogenética suele formular estos modelos en tiempo continuo, $P(t)=e^{Qt}$, con un generador $Q$; el enfoque en tiempo discreto que adoptamos se relaciona con aquél mediante $P=e^{Q\Delta t}$. La pregunta inversa —cuándo una matriz estocástica observada proviene de un generador, o admite raíces que sean a su vez estocásticas— es el problema de inmersión (*embedding problem*) planteado por Kingman (1962), y resulta central cuando se pretende recuperar la matriz de un paso a partir de datos acumulados durante millones de años.

Los modelos de sustitución forman una jerarquía según las restricciones que imponen sobre $P$ (o $Q$). El más simple, de Jukes y Cantor (1969), supone que todos los cambios entre nucleótidos son equiprobables y conduce a la conocida corrección de la distancia observada $p$ por sustituciones múltiples, $d=-\tfrac{3}{4}\ln\!\left(1-\tfrac{4}{3}p\right)$. Sin embargo, la bioquímica del ADN no es simétrica: las bases se dividen en purinas ($A,G$) y pirimidinas ($C,T$), y las **transiciones** (cambios dentro de cada clase, $A\leftrightarrow G$ y $C\leftrightarrow T$) son mucho más frecuentes que las **transversiones** (cambios entre clases). Kimura (1980) incorporó esta asimetría con un modelo de dos parámetros; Hasegawa, Kishino y Yano (1985) añadieron frecuencias de equilibrio desiguales y lo aplicaron precisamente a datar la separación entre humanos y grandes simios con ADN mitocondrial; y Tavaré (1986) formuló el modelo reversible en el tiempo más general, base de la inferencia filogenética por máxima verosimilitud (Felsenstein, 1981).

El ADN mitocondrial (ADNmt) es especialmente adecuado para este tipo de análisis en primates. Brown, George y Wilson (1979) mostraron que evoluciona del orden de diez veces más rápido que la fracción de copia única del genoma nuclear, por lo que acumula diferencias medibles en escalas de pocos millones de años. Al secuenciar un segmento de ADNmt de humano, chimpancé, gorila, orangután y gibón, Brown et al. (1982) documentaron además un fuerte sesgo hacia las transiciones, cuya proporción disminuye a medida que aumenta la distancia evolutiva: una señal de **saturación**, es decir, de sustituciones múltiples en un mismo sitio que dejan de ser visibles en la comparación directa. Dentro del ADNmt, el gen del citocromo *b* (MT-CYB, de unos 1140 pb) se ha usado ampliamente como marcador filogenético en mamíferos desde el trabajo de Irwin, Kocher y Wilson (1991). Para calibrar el modelo, los tiempos de divergencia de referencia se toman de TimeTree 5, una síntesis de los tiempos de divergencia publicados en miles de estudios (Kumar et al., 2022).

El mismo principio se aplica a proteínas, con un espacio de 20 estados. Las matrices PAM de Dayhoff, Schwartz y Orcutt (1978) se construyen estimando una matriz de transición de un paso (PAM1, que corresponde a una mutación aceptada por cada 100 residuos) a partir de proteínas muy emparentadas, y obteniendo PAM$n$ como su $n$-ésima potencia: una cadena de Markov homogénea en tiempo discreto en sentido estricto. Las matrices BLOSUM (Henikoff y Henikoff, 1992), en cambio, se estiman directamente de bloques conservados con distintos niveles de identidad, sin extrapolar mediante potencias. Estas matrices de sustitución son hoy la base de los algoritmos de alineamiento y búsqueda en bases de datos de secuencias.

En este trabajo modelamos la evolución de cada sitio del gen MT-CYB como una cadena de Markov homogénea en tiempo discreto sobre $S=\{A,G,C,T\}$. A partir de las secuencias de seis primates —humano, chimpancé, gorila, orangután, gibón y macaco—, usamos al humano como referencia, estimamos la matriz de transición de un paso a partir de las comparaciones pareadas y de los tiempos de divergencia de TimeTree, y evaluamos la capacidad del modelo para recuperar dichos tiempos a partir de las diferencias observadas. Discutimos además las limitaciones del enfoque: la saturación en las divergencias más profundas, la heterogeneidad de tasas entre sitios y linajes, y las dificultades del problema de inmersión al extraer raíces de matrices estocásticas empíricas.


## 2. Problema y objetivos.

**Problema:** Dada dos secuencias de nucleótidos que divergeron de un ancestro común hace $n$ generaciones, ¿cómo modelar el cambio en cada sitio y cómo estimar $n$ a partir de las diferencias observadas?

**Objetivo general:** Estudiar la sustitución de nucleótidos en un sitio del ADN como una cadena de Markov finita y homogénea —analizando su irreducibilidad, aperiodicidad, distribución estacionaria y reversibilidad— e implementar un modelo computacional que permita estimar tiempos de divergencia en primates a partir del gen MT-CYB, caracterizando los alcances y límites del reloj molecular.

**Objetivos específicos:**
1. Formular el modelo por sitio y demostrar que, si el ancestro común está en la distribución estacionaria $\pi$, la cadena es reversible y ambas ramas evolucionan de forma independiente con la misma matriz $P$, la distribución conjunta de las bases en dos especies actuales es $F_n=\Pi P^{2n}$, con $\Pi=\operatorname{diag}(\pi)$; en particular, $F_n$ es simétrica.
2. Contrastar empíricamente los supuestos del modelo en el alineamiento: simetría de la matriz de conteos (prueba de Bowker, que contrasta la homogeneidad del proceso entre linajes), estabilidad de la composición de bases entre especies y heterogeneidad de tasas por posición del codón.
3. Estimar la matriz de un paso $P$ como raíz espectral de la matriz de divergencia simetrizada, verificar si es estocástica, irreducible y aperiódica, y analizar los casos en que no existe una raíz estocástica (problema de inmersión).
4. Demostrar que la identidad esperada $p(m)=\sum_i \pi_i (P^m)_{ii}$ es estrictamente decreciente y converge a $\sum_i \pi_i^2$, y usarla para estimar los tiempos de divergencia, validando con la estrategia de dejar una especie fuera y contrastando con TimeTree.
5. Relacionar la saturación con la convergencia a la distribución estacionaria, cuantificar el horizonte temporal del reloj y verificar por simulación (funciones de iniciación $\psi$ y de actualización $\phi$) el sesgo y la varianza del estimador según el tiempo de divergencia.
6. Cuantificar la incertidumbre del tiempo de divergencia mediante el algoritmo de Metropolis sobre el espacio finito de pasos $n$, verificando su convergencia contra la distribución a posteriori exacta.

## 3. Marco teórico.

Seguimos la notación y los resultados de Häggström (2002), capítulos 2 a 7. Los resultados del libro se citan con su numeración original (por ejemplo, *Teorema 5.2* es el teorema de convergencia de cadenas de Markov). Los resultados propios del trabajo se numeran como *Lema*, *Proposición* y *Corolario* 1, 2, …, con demostración.

**Notación.** $\mathbf 1=(1,1,1,1)^\top$; para un vector de probabilidad $\pi$ con entradas positivas, $\Pi=\operatorname{diag}(\pi_1,\dots,\pi_4)$, $\Pi^{1/2}=\operatorname{diag}(\sqrt{\pi_1},\dots,\sqrt{\pi_4})$ y $\sqrt{\pi}=(\sqrt{\pi_1},\dots,\sqrt{\pi_4})^\top$. Para una matriz estocástica $P$, $P^m(i,\cdot)$ denota la fila $i$ de $P^m$.

### 3.1. La cadena de sustitución en un sitio.

**Espacio de estados.** $S=\{s_1,s_2,s_3,s_4\}=\{A,G,C,T\}$, ordenado de modo que las purinas ($A,G$) preceden a las pirimidinas ($C,T$). Con este orden, las transiciones ($A\leftrightarrow G$, $C\leftrightarrow T$) ocupan los bloques diagonales $2\times 2$ de la matriz de transición, y las transversiones los bloques fuera de la diagonal.

**El proceso.** Fijamos un sitio del gen y un linaje, y discretizamos el tiempo en pasos de duración $\Delta t$. Sea $X_k\in S$ la base presente en ese sitio en el instante $k\Delta t$. Modelamos $(X_0,X_1,\dots)$ como una cadena de Markov homogénea con matriz de transición $P$ (Häggström, Definición 2.1):

$$\mathbb P(X_{k+1}=s_j\mid X_0=s_{i_0},\dots,X_{k-1}=s_{i_{k-1}},X_k=s_i)=\mathbb P(X_{k+1}=s_j\mid X_k=s_i)=P_{i,j}.$$

La propiedad de Markov expresa que la probabilidad de mutar depende sólo de la base actual; la homogeneidad (que $P$ no dependa de $k$) es la formulación probabilística del **reloj molecular**. Por el Teorema 2.1, si $\mu^{(0)}$ es la distribución inicial, la distribución en el instante $k$ es $\mu^{(k)}=\mu^{(0)}P^k$, y $(P^k)_{i,j}=\mathbb P(X_k=s_j\mid X_0=s_i)$.

**Dos especies actuales.** Consideramos un ancestro común en el instante $0$, cuya base en el sitio es $Z$. A partir de él surgen dos linajes, $(X^{(1)}_k)_{k=0}^{n_1}$ y $(X^{(2)}_k)_{k=0}^{n_2}$, con $X^{(1)}_0=X^{(2)}_0=Z$. Las bases observadas hoy en las dos especies son $X:=X^{(1)}_{n_1}$ e $Y:=X^{(2)}_{n_2}$. El ancestro no se observa. Trabajamos bajo las siguientes hipótesis:

- **(H1)** Cada linaje es una cadena de Markov homogénea con **la misma** matriz $P$.
- **(H2)** Condicionalmente a $Z$, los dos linajes evolucionan de forma independiente.
- **(H3)** $Z\sim\pi$, donde $\pi$ es una distribución estacionaria de $P$ (Definición 5.1).
- **(H4)** $P$ es reversible respecto de $\pi$ (Definición 6.1).
- **(H5)** Los $L$ sitios del alineamiento son copias independientes e idénticamente distribuidas del par $(X,Y)$.
- **(H6)** $P_{i,i}>\tfrac12$ para todo $i$.

(H6) es una condición sobre la escala temporal. Basta tomar $\Delta t$ suficientemente pequeño: con $\Delta t=0{,}1$ Ma, la probabilidad de que un sitio de MT-CYB cambie en un paso es del orden de $10^{-3}$. Cuando hablemos de "tiempo de divergencia" supondremos además $n_1=n_2=n$; esto es (H1) aplicada al reloj: ambos linajes transcurren el mismo tiempo desde la separación.

### 3.2. Irreducibilidad, aperiodicidad y distribución estacionaria.

Recordemos que la cadena es **irreducible** si para todo par $s_i,s_j$ existe $m$ con $(P^m)_{i,j}>0$ (Definición 4.1), y que el período de $s_i$ es $d(s_i)=\gcd\{m\ge 1:(P^m)_{i,i}>0\}$, siendo la cadena **aperiódica** si $d(s_i)=1$ para todo $i$ (Definición 4.2).

**Lema 1.** *(a) Si $P_{i,i}>0$ para todo $i$, la cadena es aperiódica; en particular, (H6) implica aperiodicidad. (b) Si $P_{i,j}>0$ para todo $i,j$, la cadena es irreducible. (c) Si la cadena es irreducible y $\pi$ es estacionaria, entonces $\pi_i>0$ para todo $i$.*

*Demostración.* (a) Si $P_{i,i}>0$, entonces $1\in\{m\ge1:(P^m)_{i,i}>0\}$ y el máximo común divisor de ese conjunto es $1$. (b) Basta tomar $m=1$. (c) Como $\sum_i\pi_i=1$, existe $l$ con $\pi_l>0$. Dado $j$, por irreducibilidad existe $m$ con $(P^m)_{l,j}>0$. Como $\pi=\pi P^m$, se tiene $\pi_j=\sum_i\pi_i(P^m)_{i,j}\ge\pi_l(P^m)_{l,j}>0$. $\blacksquare$

La condición (b) es biológicamente natural: en un paso, cualquier base puede sustituirse por cualquier otra con probabilidad positiva, aunque sea pequeña. En adelante suponemos que $P$ es irreducible. Bajo irreducibilidad y aperiodicidad se aplican los tres resultados centrales del capítulo 5:

- *Teorema 5.1 (existencia):* existe al menos una distribución estacionaria $\pi$.
- *Teorema 5.3 (unicidad):* esa distribución es única.
- *Teorema 5.2 (convergencia):* para cualquier distribución inicial $\mu^{(0)}$, $\;d_{TV}(\mu^{(k)},\pi)\to 0$ cuando $k\to\infty$, donde $d_{TV}(\nu,\nu')=\tfrac12\sum_i\lvert\nu_i-\nu'_i\rvert$ (Definición 5.2).

**Interpretación.** Tomando $\mu^{(0)}=\delta_{s_i}$, el Teorema 5.2 dice que $P^k(i,\cdot)\to\pi$ para cada $i$: tras un tiempo largo, la base presente en el sitio es prácticamente independiente de la base ancestral. Ésta es la formulación matemática de la **saturación**. La diferencia entre dos secuencias deja de crecer con el tiempo, y con ella se pierde la información que el reloj necesita. Además, bajo (H3) la composición de bases de cada especie actual es $\mu^{(n)}=\pi P^n=\pi$: todas las especies deben tener, en promedio, la misma composición, igual a $\pi$. Esta consecuencia se puede contrastar con los datos.

### 3.3. Reversibilidad.

Una distribución $\pi$ es **reversible** para $P$ si $\pi_iP_{i,j}=\pi_jP_{j,i}$ para todo $i,j$ (Definición 6.1). Por el Teorema 6.1, toda distribución reversible es estacionaria. En forma matricial, la condición de reversibilidad dice que $\Pi P$ es simétrica, es decir, $\Pi P=P^\top\Pi$.

**Lema 2.** *Si $P$ es reversible respecto de $\pi$, entonces $P^m$ es reversible respecto de $\pi$ para todo $m\ge 0$; es decir, $\Pi P^m$ es simétrica.*

*Demostración.* Aplicando $m$ veces la identidad $\Pi P=P^\top\Pi$:
$$\Pi P^m=(\Pi P)P^{m-1}=P^\top\Pi P^{m-1}=(P^\top)^2\Pi P^{m-2}=\cdots=(P^\top)^m\Pi=(\Pi P^m)^\top.\qquad\blacksquare$$

**Cadena invertida en el tiempo.** Si $\pi$ es estacionaria y positiva, definimos $\tilde P=\Pi^{-1}P^\top\Pi$, es decir, $\tilde P_{i,j}=\pi_jP_{j,i}/\pi_i$. Es una matriz estocástica, porque $\sum_j\pi_jP_{j,i}=\pi_i$. Su interpretación es la siguiente: si $X_0\sim\pi$, entonces $\mathbb P(X_k=s_j\mid X_{k+1}=s_i)=\tilde P_{i,j}$, de modo que $\tilde P$ describe la cadena recorrida hacia atrás en el tiempo. La condición $P$ reversible equivale a $\tilde P=P$: la cadena "se ve igual" hacia adelante y hacia atrás (Häggström, Problema 6.3).

### 3.4. Distribución conjunta de las bases en dos especies actuales.

Éste es el resultado que conecta el modelo con los datos: lo que observamos no es una trayectoria de la cadena, sino el par $(X,Y)$ en los extremos de dos ramas.

**Proposición 1.** *Bajo (H1)–(H3), con ramas de $n_1$ y $n_2$ pasos,*
$$\mathbb P(X=s_i,\;Y=s_j)=\big((P^{n_1})^\top\,\Pi\,P^{n_2}\big)_{i,j}.$$
*Si además vale (H4), entonces*
$$\mathbb P(X=s_i,\;Y=s_j)=\pi_i\,(P^{n_1+n_2})_{i,j},\qquad\text{es decir,}\qquad F:=\big(\mathbb P(X=s_i,Y=s_j)\big)_{i,j}=\Pi P^{n_1+n_2}.$$

*Demostración.* Por la ley de probabilidad total sobre el valor de $Z$, la independencia condicional (H2), y el Teorema 2.1 junto con (H1) y (H3):
$$\mathbb P(X=s_i,Y=s_j)=\sum_{k}\mathbb P(Z=s_k)\,\mathbb P(X=s_i\mid Z=s_k)\,\mathbb P(Y=s_j\mid Z=s_k)=\sum_k\pi_k\,(P^{n_1})_{k,i}\,(P^{n_2})_{k,j},$$
que es la entrada $(i,j)$ de $(P^{n_1})^\top\Pi P^{n_2}$. Bajo (H4), el Lema 2 da $\pi_k(P^{n_1})_{k,i}=\pi_i(P^{n_1})_{i,k}$, de donde
$$\mathbb P(X=s_i,Y=s_j)=\pi_i\sum_k(P^{n_1})_{i,k}(P^{n_2})_{k,j}=\pi_i\,(P^{n_1+n_2})_{i,j}.\qquad\blacksquare$$

**Corolario 1.** *Bajo (H1)–(H4) con $n_1=n_2=n$, sea $F_n:=\Pi P^{2n}$. Entonces:*
1. *$F_n$ es simétrica, y ambas marginales son $\pi$: $F_n\mathbf 1=\pi^\top$ y $\mathbf 1^\top F_n=\pi$.*
2. *$\mathbb P(Y=s_j\mid X=s_i)=(P^{2n})_{i,j}$. Condicionando a la base actual de una especie, la otra se comporta como el estado de **una sola** cadena de $2n$ pasos que parte de esa base. Es el "principio de la polea" de Felsenstein (1981).*
3. *$F$ depende de $(n_1,n_2)$ sólo a través de $n_1+n_2$. Con dos especies, la posición del ancestro sobre la rama no es identificable; lo que permite hablar de "tiempo desde la divergencia", $(n_1+n_2)/2$, es el supuesto de reloj $n_1=n_2$.*

*Demostración.* 1. La simetría es el Lema 2 con $m=2n$. Las marginales salen de que $P^{2n}\mathbf 1=\mathbf 1$ y $\pi P^{2n}=\pi$. 2. $\mathbb P(Y=s_j\mid X=s_i)=\pi_i(P^{2n})_{i,j}/\pi_i$, usando el punto 1 y $\pi_i>0$ (Lema 1(c)). 3. Es inmediato de la Proposición 1. $\blacksquare$

**Observación (qué se puede contrastar y qué no).** Con $n_1=n_2$, la primera fórmula de la Proposición 1 muestra que $F=(P^n)^\top\Pi P^n$ es simétrica **para cualquier $P$, sea reversible o no**. Por lo tanto, una prueba de simetría sobre los conteos (Bowker, 1948) contrasta (H1)–(H3): mismo proceso en ambos linajes, ramas iguales y ancestro estacionario. No contrasta la reversibilidad. Lo que la reversibilidad aporta es la forma $\Pi^{-1}F=P^{2n}$. En general, $\Pi^{-1}F=\Pi^{-1}(P^n)^\top\Pi\,P^n=\tilde P^{\,n}P^n$: se retrocede $n$ pasos con la cadena invertida y se avanza $n$ pasos con la original. Ambas expresiones coinciden exactamente cuando $\tilde P=P$. Por su parte, la igualdad de las composiciones de bases entre especies contrasta (H3).

**Proposición 2 (estimador de máxima verosimilitud de $F$).** *Sea $N=(N_{i,j})$ la matriz de conteos del alineamiento, donde $N_{i,j}$ es el número de sitios con $s_i$ en la especie 1 y $s_j$ en la especie 2, y $L=\sum_{i,j}N_{i,j}$. Bajo (H5), $N\sim\text{Multinomial}(L,F)$. El máximo de la verosimilitud sobre las matrices de probabilidad simétricas se alcanza en*
$$\hat F_{i,j}=\frac{N_{i,j}+N_{j,i}}{2L}.$$

*Demostración.* La log-verosimilitud es $\ell(F)=\sum_{i,j}N_{i,j}\log F_{i,j}$. Si $F$ es simétrica, sus parámetros libres son $f_{ii}$ y $f_{ij}=F_{i,j}=F_{j,i}$ para $i<j$, y
$$\ell=\sum_i N_{i,i}\log f_{ii}+\sum_{i<j}(N_{i,j}+N_{j,i})\log f_{ij},\qquad\text{sujeto a}\quad \sum_i f_{ii}+2\sum_{i<j}f_{ij}=1.$$
Con un multiplicador de Lagrange $\lambda$, las condiciones de primer orden son $N_{i,i}/f_{ii}=\lambda$ y $(N_{i,j}+N_{j,i})/f_{ij}=2\lambda$. Sustituyendo en la restricción se obtiene $\lambda=L$. Como $\ell$ es estrictamente cóncava en los $f$, el punto crítico es el máximo. $\blacksquare$

Por la Ley de los Grandes Números (Häggström, Teorema 1.2), aplicada a los indicadores $\mathbf 1\{X=s_i,Y=s_j\}$ de cada sitio, $\hat F\to F$ en probabilidad cuando $L\to\infty$.

### 3.5. Estructura espectral de una cadena reversible.

**Lema 3.** *Si $P$ es reversible respecto de $\pi>0$, la matriz $A:=\Pi^{1/2}P\,\Pi^{-1/2}$ es simétrica. En consecuencia, existen una matriz ortogonal $U$ y valores propios reales $\lambda_1,\dots,\lambda_4$ tales que $A=U\Lambda U^\top$, con $\Lambda=\operatorname{diag}(\lambda_k)$, y*
$$P^m=\Pi^{-1/2}\,U\Lambda^mU^\top\,\Pi^{1/2},\qquad (P^m)_{i,j}=\sqrt{\tfrac{\pi_j}{\pi_i}}\;\sum_{k=1}^{4}\lambda_k^m\,U_{ik}U_{jk}\qquad(m\ge 0).$$

*Demostración.* $A_{i,j}=\sqrt{\pi_i}P_{i,j}/\sqrt{\pi_j}=\pi_iP_{i,j}/\sqrt{\pi_i\pi_j}$, que es simétrica por la Definición 6.1. Por el teorema espectral, $A=U\Lambda U^\top$. Además $P=\Pi^{-1/2}A\Pi^{1/2}$, así que $P^m=\Pi^{-1/2}A^m\Pi^{1/2}$. $\blacksquare$

**Lema 4.** *Bajo las hipótesis del Lema 3:*
- *(a) $A\sqrt\pi=\sqrt\pi$, y $\sqrt\pi$ es unitario;*
- *(b) $\lvert\lambda_k\rvert\le 1$ para todo $k$;*
- *(c) si además vale (H6), entonces $\lambda_k>0$ para todo $k$.*

*Demostración.* (a) $A\sqrt\pi=\Pi^{1/2}P\,\Pi^{-1/2}\sqrt\pi=\Pi^{1/2}P\mathbf 1=\Pi^{1/2}\mathbf 1=\sqrt\pi$, y $\lVert\sqrt\pi\rVert^2=\sum_i\pi_i=1$. (b) $A$ y $P$ son semejantes, así que tienen los mismos valores propios. Si $Pv=\lambda v$ con $v\ne0$ y $\lvert v_i\rvert=\max_l\lvert v_l\rvert$, entonces $\lvert\lambda\rvert\lvert v_i\rvert=\lvert\sum_jP_{i,j}v_j\rvert\le\lvert v_i\rvert$. (c) Por el teorema de los círculos de Gershgorin aplicado a $P$, cada valor propio cumple $\lvert\lambda-P_{i,i}\rvert\le\sum_{j\ne i}P_{i,j}=1-P_{i,i}$ para algún $i$. Como $\lambda$ es real, $\lambda\ge 2P_{i,i}-1>0$. $\blacksquare$

**Proposición 3.** *Si $P$ es irreducible, reversible respecto de $\pi$ y satisface (H6), entonces, ordenando los valores propios de mayor a menor y tomando $U_{\cdot 1}=\sqrt\pi$,*
$$1=\lambda_1>\lambda_2\ge\lambda_3\ge\lambda_4>0.$$

*Demostración.* Por el Lema 4, $\lambda_1=1$ (con vector propio $\sqrt\pi$) y todos los $\lambda_k\in(0,1]$. Falta ver que $\lambda_k<1$ para $k\ge2$. Por el Lema 1(a), la cadena es aperiódica. Aplicando el Teorema 5.2 con $\mu^{(0)}=\delta_{s_i}$ para cada $i$, se obtiene $P^m\to\mathbf 1\pi$ (todas las filas tienden a $\pi$). Entonces
$$\Lambda^m=U^\top\Pi^{1/2}P^m\Pi^{-1/2}U\;\longrightarrow\;U^\top\Pi^{1/2}\mathbf 1\,\pi\,\Pi^{-1/2}U=U^\top\sqrt\pi\,\sqrt\pi^{\top}U=e_1e_1^\top,$$
porque $U^\top\sqrt\pi=e_1$ al ser $\sqrt\pi$ la primera columna de la matriz ortogonal $U$. Así, $\lambda_k^m\to0$ para todo $k\ge2$, lo que exige $\lambda_k<1$. $\blacksquare$

**Corolario 2 (velocidad de convergencia).** *Bajo las hipótesis de la Proposición 3, para todo $i$ y todo $m\ge0$,*
$$d_{TV}\big(P^m(i,\cdot),\pi\big)\le\frac{\lambda_2^{\,m}}{\sqrt{\pi_i}}.$$

*Demostración.* En el Lema 3, el término $k=1$ vale $\sqrt{\pi_j/\pi_i}\sqrt{\pi_i}\sqrt{\pi_j}=\pi_j$. Luego
$$\lvert(P^m)_{i,j}-\pi_j\rvert\le\sqrt{\tfrac{\pi_j}{\pi_i}}\;\lambda_2^m\sum_{k\ge2}\lvert U_{ik}\rvert\lvert U_{jk}\rvert\le\sqrt{\tfrac{\pi_j}{\pi_i}}\;\lambda_2^m,$$
donde la última desigualdad es la de Cauchy–Schwarz, ya que las filas de $U$ tienen norma 1. Sumando sobre $j$ y usando de nuevo Cauchy–Schwarz, $\sum_j\sqrt{\pi_j}\le\sqrt4\,(\sum_j\pi_j)^{1/2}=2$:
$$d_{TV}=\tfrac12\sum_j\lvert(P^m)_{i,j}-\pi_j\rvert\le\tfrac12\,\frac{\lambda_2^m}{\sqrt{\pi_i}}\cdot 2.\qquad\blacksquare$$

**Interpretación.** El Teorema 5.2 dice *que* la cadena olvida la base ancestral; el Corolario 2 dice *a qué velocidad*: geométricamente, con razón $\lambda_2$. La cantidad $\tau=-1/\log\lambda_2$ (en pasos) es la escala de tiempo en que la memoria de la base ancestral decae en un factor $e$, y fija el horizonte temporal del reloj molecular.

### 3.6. Estimación de la matriz de un paso: raíz espectral y problema de inmersión.

Por el Corolario 1, $\Pi^{-1}F_n=P^{2n}$. Para obtener la matriz de un paso a partir de una divergencia de tiempo conocido, hay que extraer una raíz $2n$-ésima.

**Proposición 4.** *Sea $F$ una matriz simétrica de probabilidades con marginal $\pi=\mathbf 1^\top F$, $\pi>0$. Sean $D:=\Pi^{-1}F$ y $B:=\Pi^{-1/2}F\,\Pi^{-1/2}$, que es simétrica. Supongamos que todos los valores propios de $B$ son positivos y escribamos $B=V\Gamma V^\top$, con $V$ ortogonal. Para $m\ge1$, sea*
$$R_m:=\Pi^{-1/2}\,V\Gamma^{1/m}V^\top\,\Pi^{1/2}.$$
*Entonces:*
1. *$R_m^{\,m}=D$;*
2. *$R_m\mathbf 1=\mathbf 1$ (las filas suman 1);*
3. *$\Pi R_m$ es simétrica, es decir, $R_m$ satisface el balance detallado respecto de $\pi$;*
4. *$R_m$ es la única matriz $R$ con $R^m=D$ tal que $\Pi^{1/2}R\,\Pi^{-1/2}$ es simétrica definida positiva;*
5. *si $F=\Pi P^{2n}$, con $P$ reversible respecto de $\pi$ y que satisface (H6), entonces $R_{2n}=P$.*

*En cambio, $R_m$ **no** es necesariamente no negativa.*

*Demostración.* 1. $R_m^m=\Pi^{-1/2}V\Gamma V^\top\Pi^{1/2}=\Pi^{-1/2}B\,\Pi^{1/2}=D$. 2. $B\sqrt\pi=\Pi^{-1/2}F\mathbf 1=\Pi^{-1/2}\pi^\top=\sqrt\pi$, así que $\sqrt\pi$ es vector propio de $B$ con valor propio $1$, y por lo tanto también de $B^{1/m}=V\Gamma^{1/m}V^\top$ con valor propio $1^{1/m}=1$. Luego $R_m\mathbf 1=\Pi^{-1/2}B^{1/m}\sqrt\pi=\Pi^{-1/2}\sqrt\pi=\mathbf 1$. 3. $\Pi R_m=\Pi^{1/2}B^{1/m}\Pi^{1/2}$ es simétrica. 4. Si $R^m=D$ y $C:=\Pi^{1/2}R\,\Pi^{-1/2}$ es simétrica definida positiva, entonces $C^m=\Pi^{1/2}D\,\Pi^{-1/2}=B$. Toda matriz simétrica definida positiva tiene una única raíz $m$-ésima simétrica definida positiva, así que $C=B^{1/m}$ y $R=R_m$. 5. Por los Lemas 3 y 4(c), $A=\Pi^{1/2}P\Pi^{-1/2}$ es simétrica definida positiva y $A^{2n}=\Pi^{1/2}P^{2n}\Pi^{-1/2}=B$. Por la unicidad del punto 4, $P=R_{2n}$. $\blacksquare$

**El problema de inmersión.** Sea $Q:=\Pi^{-1/2}V(\log\Gamma)V^\top\Pi^{1/2}$, el logaritmo principal (real) de $D$, de modo que $R_m=\exp(Q/m)$ y $Q\mathbf 1=0$. Desarrollando la exponencial,
$$R_m=I+\tfrac1m Q+O(m^{-2}).$$
De aquí: si $Q_{i,j}<0$ para algún $i\ne j$, entonces $R_m$ tiene una entrada negativa para todo $m$ suficientemente grande; si $Q_{i,j}>0$ para todo $i\ne j$, entonces $R_m$ es estocástica para todo $m$ suficientemente grande. Es decir, al refinar el paso temporal, la raíz es una matriz de transición esencialmente si y sólo si $Q$ es un generador de una cadena en tiempo continuo ($D=e^{Q}$). Ésta es la versión, para nuestro problema, del **problema de inmersión** de Kingman (1962). Verbyla et al. (2013) lo estudiaron empíricamente en alineamientos reales. En la práctica, una estimación $\hat F$ con pares de bases de conteo cero puede producir un $Q$ con entradas negativas fuera de la diagonal y, por lo tanto, una raíz con entradas ligeramente negativas.

**Relación con el código.** La expresión `expm(logm(D)/m)` usada en el notebook coincide con $R_m$ cuando $D$ se calcula a partir de la matriz **simetrizada** $\hat F$ de la Proposición 2. Si se aplica a la matriz de conteos sin simetrizar, el resultado no es necesariamente reversible y los puntos 3 a 5 dejan de valer.

**Identificabilidad.** $F$ depende de $(P,n)$ sólo a través de $P^{2n}$. Por el punto 4, para cualquier otro $n'$, el par $(R_{2n'},n')$ produce la misma $F$. Por lo tanto, **con una sola comparación el tiempo no es identificable**: sólo lo es la "distancia" $P^{2n}$. Para asignar tiempo absoluto hay que calibrar, es decir, fijar $P$ con divergencias de edad conocida, y luego estimar $n$ para otras especies. Esto justifica formalmente la calibración y la validación dejando una especie fuera.

### 3.7. Identidad esperada, estimador del tiempo y horizonte del reloj.

Para $m\ge0$ definimos la **identidad esperada**
$$p(m):=\sum_i\pi_i\,(P^m)_{i,i}=\operatorname{tr}(\Pi P^m).$$
Por la Proposición 1, si las dos ramas suman $m=n_1+n_2$ pasos, $p(m)=\mathbb P(X=Y)$ es la probabilidad de que un sitio tenga la misma base en ambas especies. La definición se extiende a $m\ge0$ real mediante $\lambda_k^m=e^{m\log\lambda_k}$, que tiene sentido porque $\lambda_k>0$.

**Proposición 5.** *Si $P$ es irreducible, reversible respecto de $\pi$ y satisface (H6), entonces*
$$p(m)=\sum_{k=1}^4 w_k\,\lambda_k^m,\qquad w_k:=\sum_i\pi_iU_{ik}^2\ge0,\qquad w_1=\sum_i\pi_i^2=:p_\infty,\qquad\sum_k w_k=1.$$
*En consecuencia:*
- *(a) $p(0)=1$, y $p$ es estrictamente decreciente y convexa en $[0,\infty)$;*
- *(b) $0<p(m)-p_\infty\le(1-p_\infty)\,\lambda_2^{\,m}$; en particular, $p(m)\to p_\infty$;*
- *(c) $p$ es una biyección continua de $[0,\infty)$ sobre $(p_\infty,1]$.*

*Demostración.* Por el Lema 3, $(P^m)_{i,i}=\sum_k\lambda_k^mU_{ik}^2$. Multiplicando por $\pi_i$ y sumando en $i$ se obtiene la fórmula. Además $U_{i1}=\sqrt{\pi_i}$, de donde $w_1=\sum_i\pi_i^2$, y $\sum_kw_k=\sum_i\pi_i\sum_kU_{ik}^2=\sum_i\pi_i=1$, porque las filas de $U$ tienen norma 1. Por la Proposición 3, $\lambda_k\in(0,1)$ para $k\ge2$. Por el Lema 1(c), $\pi_i>0$ para todo $i$, así que $\pi$ no es degenerada y $\sum_{k\ge2}w_k=1-p_\infty>0$.

(a) $p(0)=\sum_kw_k=1$. Además $p'(m)=\sum_{k\ge2}w_k\lambda_k^m\log\lambda_k<0$ y $p''(m)=\sum_{k\ge2}w_k\lambda_k^m(\log\lambda_k)^2>0$.

(b) $p(m)-p_\infty=\sum_{k\ge2}w_k\lambda_k^m$, que es positivo y está acotado por $\lambda_2^m\sum_{k\ge2}w_k$.

(c) Se sigue de (a), (b) y la continuidad de $p$. $\blacksquare$

**Interpretación.** $p_\infty$ es la probabilidad de que dos bases elegidas **independientemente** según $\pi$ coincidan. La Proposición 5(b) es la saturación, ahora cuantificada: la identidad entre dos especies decae hacia la identidad entre dos secuencias "aleatorias" a velocidad geométrica $\lambda_2$, la misma del Corolario 2.

**Ejemplo (Jukes–Cantor en tiempo discreto).** Sea $P_{i,j}=\alpha$ para $i\ne j$ y $P_{i,i}=1-3\alpha$, con $0<\alpha<1/6$ para que valga (H6). Entonces $\pi$ es uniforme, los valores propios son $1$ y $1-4\alpha$ (este último triple), y $p(m)=\tfrac14+\tfrac34(1-4\alpha)^m$. Si $\hat D=1-\hat p$ es la proporción observada de sitios distintos, despejando $m$ se obtiene $\hat m=\log(1-\tfrac43\hat D)/\log(1-4\alpha)$. El número esperado de sustituciones por sitio es $3\alpha\hat m$, y con $\log(1-4\alpha)\approx-4\alpha$ para $\alpha$ pequeño queda $3\alpha\hat m\approx-\tfrac34\log(1-\tfrac43\hat D)$. Es exactamente la distancia de Jukes y Cantor (1969). El modelo general contiene así el clásico como caso particular.

**Estimador del tiempo.** Con $P$ calibrada, sea $\hat p=\tfrac1L\sum_{s=1}^L\mathbf 1\{X_s=Y_s\}=\operatorname{tr}(N)/L$ la proporción de sitios idénticos. Definimos
$$\hat m=p^{-1}(\hat p)\quad\text{si }\hat p\in(p_\infty,1],\qquad \hat n=\hat m/2,\qquad \hat t=\hat n\,\Delta t.$$
Si $\hat p\le p_\infty$, las secuencias están **saturadas** y el tiempo no se puede estimar.

**Proposición 6 (consistencia).** *Bajo (H1)–(H6), con $n_1=n_2=n$ y $m=2n$, para todo $\varepsilon>0$ se tiene $\mathbb P(\lvert\hat m-m\rvert\ge\varepsilon)\to0$ cuando $L\to\infty$.*

*Demostración.* Por (H5) y la Proposición 1, los indicadores $\mathbf 1\{X_s=Y_s\}$ son i.i.d. Bernoulli de parámetro $p(m)$. Por la Ley de los Grandes Números (Teorema 1.2), $\hat p\to p(m)$ en probabilidad. Como $p(m)>p_\infty$, la probabilidad de que $\hat p\le p_\infty$ tiende a $0$. En $(p_\infty,1]$ la función $p^{-1}$ es continua (Proposición 5(c)). Por lo tanto, dado $\varepsilon>0$, existe $\delta>0$ tal que $\lvert\hat p-p(m)\rvert<\delta$ implica $\lvert\hat m-m\rvert<\varepsilon$. $\blacksquare$

**Horizonte del reloj.** Como $\operatorname{Var}(\hat p)=p(1-p)/L$, el método delta da la aproximación
$$\operatorname{Var}(\hat m)\approx\frac{p(m)\,(1-p(m))}{L\,p'(m)^2}.$$
Por la Proposición 5, $\lvert p'(m)\rvert\le(1-p_\infty)\,\lvert\log\lambda_4\rvert\,\lambda_2^{\,m}$, mientras que $p(m)(1-p(m))\to p_\infty(1-p_\infty)>0$. Luego el error estándar de $\hat m$ crece **al menos exponencialmente**, como $\lambda_2^{-m}$. Para una longitud de alineamiento $L$ dada existe un tiempo más allá del cual el reloj deja de ser informativo; cuantificarlo con nuestros datos es uno de los objetivos del trabajo.

**Estimador de máxima verosimilitud.** $\hat p$ sólo usa la diagonal de $N$. Bajo (H5), la información completa está en la log-verosimilitud
$$\ell(n)=\sum_{i,j}N_{i,j}\log\big(\pi_i\,(P^{2n})_{i,j}\big),$$
que es la que usaremos en la sección 3.9.

### 3.8. Simulación de la cadena.

Siguiendo el capítulo 3, una **función de iniciación válida** para una distribución $\mu$ es $\psi_\mu:[0,1]\to S$, con $\psi_\mu(u)=s_i$ si $u\in\big[\sum_{l<i}\mu_l,\sum_{l\le i}\mu_l\big)$. Una **función de actualización válida** para $P$ es $\phi:S\times[0,1]\to S$, con $\phi(s_i,u)=s_j$ si $u\in\big[\sum_{l<j}P_{i,l},\sum_{l\le j}P_{i,l}\big)$. Si $U_0,U_1,\dots$ son i.i.d. uniformes en $[0,1]$, entonces $X_0=\psi_\mu(U_0)$, $X_{k+1}=\phi(X_k,U_{k+1})$ es una cadena de Markov con distribución inicial $\mu$ y matriz $P$ (Häggström, ecuaciones (17)–(21)).

**Simulación de un sitio en dos especies.** Sean $U_0,U_1,\dots,U_{n_1},U'_1,\dots,U'_{n_2}$ i.i.d. uniformes. Definimos:
$$Z=\psi_\pi(U_0),\qquad X^{(1)}_0=X^{(2)}_0=Z,\qquad X^{(1)}_k=\phi(X^{(1)}_{k-1},U_k),\qquad X^{(2)}_k=\phi(X^{(2)}_{k-1},U'_k).$$
Como los dos linajes usan números aleatorios distintos e independientes, (H1)–(H3) se cumplen por construcción. Repitiendo el procedimiento con números independientes para cada uno de los $L$ sitios se obtiene (H5). Si $P$ es reversible, la Proposición 1 garantiza que el alineamiento simulado tiene la distribución conjunta $\Pi P^{n_1+n_2}$.

Como sólo interesan los extremos de las ramas, es equivalente, y mucho más rápido, simular $X^{(1)}_{n_1}=\phi^{(n_1)}(Z,U_1)$ con una función de actualización válida $\phi^{(n_1)}$ para la matriz estocástica $P^{n_1}$. La simulación se usará para: (i) verificar empíricamente la Proposición 1; (ii) estudiar el sesgo y la varianza de $\hat m$ en función del tiempo verdadero, contrastando con la aproximación del método delta; y (iii) estimar el horizonte del reloj.

### 3.9. Inferencia bayesiana del tiempo de divergencia mediante Metropolis.

Con $P$ calibrada (y por lo tanto $\pi$ fija), el parámetro desconocido es el número de pasos $n$ de cada rama. Tomamos como espacio de estados el conjunto finito $T=\{1,\dots,n_{\max}\}$ y una distribución a priori uniforme sobre $T$. Por (H5) y la Proposición 1, la verosimilitud es $\mathcal L(n)=\prod_{i,j}\big(\pi_i(P^{2n})_{i,j}\big)^{N_{i,j}}$, y la distribución a posteriori es
$$\pi^{\text{post}}(n)=\frac{\mathcal L(n)}{\sum_{n'\in T}\mathcal L(n')},\qquad n\in T.$$

**La cadena de Metropolis.** Siguiendo el capítulo 7, sea $G$ el grafo camino $1-2-\cdots-n_{\max}$, con grados $d_1=d_{n_{\max}}=1$ y $d_n=2$ en los demás vértices. La cadena de Metropolis $M$ para $\pi^{\text{post}}$ tiene probabilidades de transición (Häggström, ecuación (46))
$$M_{n,n'}=\frac1{d_n}\min\Big\{\frac{\pi^{\text{post}}(n')\,d_n}{\pi^{\text{post}}(n)\,d_{n'}},\,1\Big\}\quad\text{si }\lvert n-n'\rvert=1,\qquad M_{n,n}=1-\sum_{n'\sim n}M_{n,n'}.$$
Usamos su versión *perezosa* $M'=\tfrac12(I+M)$: en cada paso, con probabilidad $\tfrac12$ la cadena se queda donde está y, en otro caso, ejecuta un paso de $M$. El cociente $\pi^{\text{post}}(n')/\pi^{\text{post}}(n)=\exp(\ell(n')-\ell(n))$ no depende de la constante de normalización y se calcula con log-verosimilitudes, para evitar problemas de desbordamiento numérico.

**Proposición 7.** *Supongamos que $P_{i,j}>0$ para todo $i,j$. Entonces $M'$ es irreducible y aperiódica, y $\pi^{\text{post}}$ es su única distribución estacionaria. En consecuencia, si $(n_k)_{k\ge0}$ es la cadena con matriz $M'$, para cualquier estado inicial $d_{TV}(\text{Ley}(n_k),\pi^{\text{post}})\to0$ cuando $k\to\infty$.*

*Demostración.*
- *Reversibilidad.* Häggström verifica (ecuación (47)) que $\pi^{\text{post}}(n)M_{n,n'}=\pi^{\text{post}}(n')M_{n',n}$. Para $n\ne n'$ se tiene $M'_{n,n'}=\tfrac12M_{n,n'}$, así que $M'$ también es reversible respecto de $\pi^{\text{post}}$. Por el Teorema 6.1, $\pi^{\text{post}}$ es estacionaria para $M'$.
- *Irreducibilidad.* Si $P>0$, entonces $P^{2n}>0$ y $\mathcal L(n)>0$ para todo $n$. Luego cada $M'_{n,n\pm1}>0$, y cualquier $n'$ es alcanzable desde $n$ recorriendo el camino.
- *Aperiodicidad.* $M'_{n,n}\ge\tfrac12>0$, y se aplica el Lema 1(a).
- *Conclusión.* La unicidad y la convergencia siguen de los Teoremas 5.3 y 5.2. $\blacksquare$

**Observaciones.**
1. Como $T$ es finito, $\pi^{\text{post}}$ también se puede calcular **exactamente** por enumeración. Esto permite validar el algoritmo: comparar el histograma de la cadena con $\pi^{\text{post}}$, midiendo $d_{TV}$, en función del número de iteraciones.
2. En este problema unidimensional el MCMC no es imprescindible. Su valor está en que el mismo esquema se extiende sin cambios al caso en que también los parámetros de $P$ son desconocidos, donde la enumeración ya no es posible. Ésta es la base de los métodos bayesianos de datación de divergencias (Yang y Rannala, 1997; Thorne et al., 1998).
3. Al fijar $P$ en su estimación de calibración se ignora la incertidumbre de $P$. Los intervalos de credibilidad obtenidos son, por lo tanto, optimistas.

### 3.10. Supuestos del modelo y lo que dice la literatura.

El modelo descansa en supuestos que la literatura ha examinado críticamente. Los resumimos aquí, indicando la hipótesis de la sección 3.1 a la que corresponde cada uno. Los contrastamos con nuestros datos en la sección 4.

| Supuesto | Evidencia en la literatura |
|---|---|
| **Propiedad de Markov** en la base observada (H1) | Se cumple en el modelo estándar (Tavaré, 1986; Rodríguez et al., 1990). Bajo el modelo covarión, en que un sitio alterna entre estados "activo" e "inactivo", la base sola deja de ser markoviana, aunque el par (base, estado) sí lo es en un espacio de estados ampliado (Tuffley y Steel, 1998). |
| **Homogeneidad temporal** (H1) | Verbyla et al. (2013) encuentran pocas violaciones (alineamientos no "embeddables"), concentradas en la tercera posición del codón y en divergencias profundas. Ho et al. (2005) muestran que las tasas estimadas dependen de la escala temporal de la calibración. |
| **Igual tasa en todos los sitios** (H5) | Las tasas varían entre sitios (Yang, 1994). Una mezcla de cadenas con velocidades distintas no es, en general, de la forma $P^n$ para una única $P$. |
| **Estacionariedad y reversibilidad** (H3, H4) | Se violan con frecuencia; en el linaje humano, por la desaminación de los dinucleótidos CpG (Squartini y Arndt, 2008). Pueden contrastarse con pruebas de simetría por pares (Bowker, 1948; Ababneh et al., 2006). Para el caso no estacionario existen las distancias de Barry y Hartigan (1987) y LogDet (Lockhart et al., 1994). |
| **Independencia entre sitios** (H5) | Hay dependencia del contexto, sobre todo en los CpG, y efectos entre codones (Siepel y Haussler, 2004). |
| **Saturación** (Prop. 5) | La proporción de transiciones disminuye con la distancia (Brown et al., 1982). La caída de la razón transiciones/transversiones con el tiempo es una medida de la saturación (Duchêne et al., 2015). |

## 4. Metodología e implementación computacional

### 4.1. Conjunto de datos.
Se usarán secuencias homólogas de Citocromo b (MT-CytB) para 6 primates con tiempos de divergencia desde el humano bien documentados en el registro fósil y TimeTree:
* Humano (*Homo sapiens*)
* Chimpancé (*Pan troglodytes*, ~6-7 Ma)
* Gorila (*Gorilla gorilla*, ~8–9 Ma)
* Orangután (*Pongo abelii*, ~13–15 Ma)
* Gibón (*Nomascus leucogenys* ~17–20 Ma)
* Macaco (*Macaca mulatta*, ~25–29 Ma)

A continuación se cargan las secuencias correspondientes con cada nombre de la especie.

In [ ]:
# @title
%%writefile alineamiento.fasta
>Homo_sapies
ATGACCCCAATACGCAAAACTAACCCCCTAATAAAATTAATTAACCACTCATTCATCGACCTCCCCACCC
CATCCAACATCTCCGCATGATGAAACTTCGGCTCACTCCTTGGCGCCTGCCTGATCCTCCAAATCACCAC
AGGACTATTCCTAGCCATGCACTACTCACCAGACGCCTCAACCGCCTTTTCATCAATCGCCCACATCACT
CGAGACGTAAATTATGGCTGAATCATCCGCTACCTTCACGCCAATGGCGCCTCAATATTCTTTATCTGCC
TCTTCCTACACATCGGGCGAGGCCTATATTACGGATCATTTCTCTACTCAGAAACCTGAAACATCGGCAT
TATCCTCCTGCTTGCAACTATAGCAACAGCCTTCATAGGCTATGTCCTCCCGTGAGGCCAAATATCATTC
TGAGGGGCCACAGTAATTACAAACTTACTATCCGCCATCCCATACATTGGGACAGACCTAGTTCAATGAA
TCTGAGGAGGCTACTCAGTAGACAGTCCCACCCTCACACGATTCTTTACCTTTCACTTCATCTTGCCCTT
CATTATTGCAGCCCTAGCAACACTCCACCTCCTATTCTTGCACGAAACGGGATCAAACAACCCCCTAGGA
ATCACCTCCCATTCCGATAAAATCACCTTCCACCCTTACTACACAATCAAAGACGCCCTCGGCTTACTTC
TCTTCCTTCTCTCCTTAATGACATTAACACTATTCTCACCAGACCTCCTAGGCGACCCAGACAATTATAC
CCTAGCCAACCCCTTAAACACCCCTCCCCACATCAAGCCCGAATGATATTTCCTATTCGCCTACACAATT
CTCCGATCCGTCCCTAACAAACTAGGAGGCGTCCTTGCCCTATTACTATCCATCCTCATCCTAGCAATAA
TCCCCATCCTCCATATATCCAAACAACAAAGCATAATATTTCGCCCACTAAGCCAATCACTTTATTGACT
CCTAGCCGCAGACCTCCTCATTCTAACCTGAATCGGAGGACAACCAGTAAGCTACCCTTTTACCATCATT
GGACAAGTAGCATCCGTACTATACTTCACAACAATCCTAATCCTAATACCAACTATCTCCCTAATTGAAA
ACAAAATACTCAAATGGGCCT

>Pan_troglodytes
ATGACCCCGACACGCAAAATTAACCCACTAATAAAATTAATTAATCACTCATTTATCGACCTCCCCACCC
CATCCAACATTTCCGCATGATGGAACTTCGGCTCACTTCTCGGCGCCTGCCTAATCCTTCAAATTACCAC
AGGATTATTCCTAGCTATACACTACTCACCAGACGCCTCAACCGCCTTCTCGTCGATCGCCCACATCACC
CGAGACGTAAACTATGGTTGGATCATCCGCTACCTCCACGCTAACGGCGCCTCAATATTTTTTATCTGCC
TCTTCCTACACATCGGCCGAGGTCTATATTACGGCTCATTTCTCTACCTAGAAACCTGAAACATTGGCAT
TATCCTCTTGCTCACAACCATAGCAACAGCCTTTATGGGCTATGTCCTCCCATGAGGCCAAATATCCTTC
TGAGGAGCCACAGTAATTACAAACCTACTGTCCGCTATCCCATACATCGGAACAGACCTGGTCCAGTGAG
TCTGAGGAGGCTACTCAGTAGACAGCCCTACCCTTACACGATTCTTCACCTTCCACTTTATCTTACCCTT
CATCATCACAGCCCTAACAACACTTCATCTCCTATTCTTACACGAAACAGGATCAAATAACCCCCTAGGA
ATCACCTCCCACTCCGACAAAATTACCTTCCACCCCTACTACACAATCAAAGATATCCTTGGCTTATTCC
TTTTCCTCCTTATCCTAATGACATTAACACTATTCTCACCAGGCCTCCTAGGCGATCCAGACAACTATAC
CCTAGCTAACCCCCTAAACACCCCACCCCACATTAAACCCGAGTGATACTTTCTATTTGCCTACACAATC
CTCCGATCCATCCCCAACAAACTAGGAGGCGTCCTCGCCCTACTACTATCTATCCTAATCCTAACAGCAA
TCCCTGTCCTCCACACATCCAAACAACAAAGCATAATATTTCGCCCACTAAGCCAACTGCTTTACTGACT
CCTAGCCACAGACCTCCTCATCCTAACCTGAATCGGAGGACAACCAGTAAGCTACCCCTTCATCACCATC
GGACAAATAGCATCCGTATTATACTTCACAACAATCCTAATCCTAATACCAATCGCCTCTCTAATCGAAA
ACAAAATACTTGAATGAACCT

>Gorilla_gorilla
ATGACCCCTATACGCAAAACTAACCCACTAGCAAAACTAATTAACCACTCATTCATTGACCTCCCTACCC
CGTCCAACATCTCCACATGATGAAACTTCGGCTCACTCCTTGGTGCCTGCTTAATCCTTCAAATCACCAC
AGGGCTATTCCTAGCCATACACTACTCACCTGATGCCTCAACCGCCTTCTCATCAATCGCCCACATCACC
CGAGATGTAAACTATGGCTGAACCATCCGCTACCTCCACGCTAACGGCGCCTCAATATTCTTCATTTGCC
TCTTTCTACACATCGGCCGGGGCCTATACTACGGCTCATTTCTCCACCAAGAAACCTGAAACATCGGCAT
CATCCTCCTACTCACAACCATAGCAACAGCCTTCATAGGCTATGTCCTCCCATGAGGCCAAATATCCTTC
TGAGGGGCCACAGTAATCACAAACTTGCTATCCGCCATCCCGTACATCGGAACAGATCTAGTCCAATGAG
TTTGAGGTGGTTACTCAGTAGATAGCCCTACCCTTACACGATTCTTTACCTTCCACTTTATCCTACCCTT
CATTATCACAGCCCTAACAACCCTCCATCTCCTATTTCTACACGAAACAGGATCAAACAACCCTCTAGGC
ATCCCCTCCCACTCTGACAAAATCACCTTCCACCCCTACTACACAATCAAAGACATCCTAGGCCTATTCT
TCTTTCTCCTGACCTTGATAACATTAACACTATTCTCACCAGACCTCCTAGGAGACCCAGACAACTACAC
TTTAGCCAACCCCCTAAACACCCCACCCCACATCAAACCCGAATGATATTTCCTATTTGCCTACGCAATT
CTCCGATCTGTCCCCAATAAACTAGGAGGCGTCTTAGCTCTATTACTATCCATTCTCATCCTAACAATAA
TTCCTATTCTCCACATATCCAAACAACAAAGCATAATATTCCGCCCATTAAGCCAACTACTCTACTGATT
CCTAATCGCAAACCTCTTCACCCTAACCTGAATCGGAGGACAACCAGTAAGCTACCCCTTCATTACCATT
GGGCAAGTAGCATCCGTACTATACTTCACGACAATCCTATTCCTGATACCAATCACATCCCTGATCGAAA
ACAAAATACTCAAATGAACCT

>Pongo_abelii
ATGACCTCAACACGTAAAACCAACCCACTAATAAAATTAATCAACCACTCACTTATCGACCTCCCCACCC
CATCAAACATCTCCGCATGATGGAACTTCGGCTCACTCCTAGGCGCCTGCTTAATCATCCAAATCACCAC
TGGACTATTCCTAGCTATACATTATTCACCAGACGCCTCCACTGCCTTTTCATCAATCGCCCACATCACT
CGAGATGTAAACTACGGCTGAATAATTCGCCACCTCCACGCTAACGGCGCCTCAATATTCTTTATCTGCC
TCTTCTTACATATCGGCCGAGGCCTATACTATGGCTCATTCACCCACCTAGAAACCTGAAACATCGGCAT
CATCCTACTATTTACAACTATAATAACAGCCTTCATAGGTTACGTCCTCCCATGAGGCCAAATATCCTTC
TGAGGAGCCACAGTAATCACAAATCTACTGTCCGCCATCCCATACATTGGAACAGACCTGGTCCAATGAG
TCTGAGGTGGCTACTCAGTAAATAGCCCCACTCTAACACGATTCTTCACCCTACACTTCATACTACCCTT
CATTATTACAGCCCTAACAACTCTACACCTCTTATTCCTACACGAAACAGGATCAAATAACCCCCTGGGA
ATCCCCTCCCATTCCGACAAAATCACCTTCCACCCCTACTACACAATCAAAGACATCCTAGGCCTACTCC
TTTTTCTCCTCGCCCTAATAACACTAACACTACTCTCACCAGACCTCCTAAGCGACCCAGACAACTACAC
CTTAGCTAACCCCCTAAGCACCCCACCCCACATTAAACCCGAATGATATTTCCTATTCGCCTACGCAATC
CTACGATCCGTCCCCAACAAACTAGGAGGTGTAATAGCCCTCATACTATCCATCCTAATCCTAACAACAA
TCCCTGCCCTTCACATGTCCAAGCAACAGAGCATAACATTTCGCCCATTGAGCCAATTCCTATATTGACT
TTTAATCGCCGACCTTCTAATTCTCACCTGAATTGGAGGGCAACCAGTAAGCTACCCCTTCATCACCATT
AGCCAAGTAGCATCCACATTGTACTTCACTACTATCCTTCTACTTATACCAGCCTCTTCCCTGATCGAAA
ACCACATACTCAAATGAACCT

>Nomascus_leucogenys
ATGACTCCCCTACGCAAAACTAACCCATTAATAAAACTAATCAATCACTCACTTGTTGACCTTCCAGCCC
CATCCAATATCTCTATATGATGAAACCTTGGCTCACTCCTAGGCACCTGCCTAATCCTCCAAATCATCAC
AGGGTTATTCCTGGCCATGCACTATACACCAGACGCCTCCATAGCCTTCTCATCAGTAGCCCACATTACC
CGAGACGTAAACTACGGCTGGGTTATCCGCTACCTTCATGCCAACGGTGCCTCAATATTTTTCATTTGCC
TATTCCTACACATCGGCCGGGGCATATATTACGGCTCATTCCTCTACTTAGAAACCTGAAACATCGGCAT
CATCCTCTTATTAGCGACCATAGCAACAGCCTTCATAGGCTACGTCCTCCCATGAGGCCAAATATCCTTC
TGAGGTGCCACAGTAATCACAAACCTACTATCTGCCGTCCCATACATCGGAACAGACCTAGTCCAATGAG
TCTGAGGCGGCTATTCAGTGGACAACGCTACACTCACACGCTTCTTTACCTTCCACTTTATCCTGCCCTT
TATCATCACAGCCCTAGTAACCCTGCACCTACTATTCTTACACGAAACAGGATCAAACAACCCCCTGGGT
ATCTCCTCCCAACCAGACAAAATCACCTTTCACCCTTACTACACAACCAAAGATATTCTAGGACTATTCC
TCCTCCTTCTCACCCTAATGAGCCTAGTACTATTCTCGCCAGACCTCCTAGGTGACCCAGACAACTACAT
CCAAGCCAACCCCCTAAGTACCCCTCCCCACATCAAGCCTGAATGGTATTTCTTATTCGCATACGCCATC
CTACGATCTGTCCCTAACAAATTGGGAGGCGTGCTAGCCCTCCTACTGTCAATCCTCATCCTAATAACAA
TCCCCATACTCCACACAGCCAAACAACAAAGCATAATATTCCGCCCACTGAGCCAGCTCACATACTGACT
CTGGGCAGCAAATCTACTAACCCTCACATGAATCGGAGGACAACCAGTGAGCTATCCATTCATCACCATC
GGACAAGTGACATCCGTACTATACTTCATCACAATCCTAATCCTAATACCAACAGCCTCTTTAATCGAAA
ACAAAATACTCAAATGGACCT

>Macaca_mulatta
ATGACTCCAATACGCAAATCCAACCCAATCCTAAAAATAATTAATCGCTCCTTCATCGATTTACCCGCCC
CACCCAACCTCTCCATATGGTGAAACTTTGGCTCACTTCTTGCAGCCTGCCTAATTTTACAAATCATCAC
AGGCCTACTCCTAGCAATACACTACTCACCAGACACCTCCTCCGCCTTCTCCTCAATTGCACATATCACC
CGAGATGTAAAGTACGGCTGAATCACTCGCTACCTCCACGCCAATGGTGCCTCTATACTCTTCATTTGCC
TTTTCCTACACATCGGTCGGGGCCTTTACTACGGCTCATACCTCCTCCTAGAAACCTGAAACATTGGTAT
TATACTCCTTCTTATAACTATAACAACGGCTTTCATGGGTTATGTTCTCCCATGAGGCCAAATATCATTC
TGGGGAGCAACAGTAATCACAAACCTGCTATCAGCAATCCCGTATATCGGAACCAATCTCGTCCAATGAA
TCTGAGGAGGATACGCCATCGACAGCCCTACTCTCACACGATTCTTCACCTTACACTTTATCCTACCCTT
CATCATCATCGCCCTCACAACCGTGCACCTACTATTCCTGCACGAAACAGGATCAAACAACCCTTGCGGA
ATCTCCTCCGACTCAGACAAAATCGCCTTCCACCCCTACTACACAACCAAAGACATCCTGGGCCTAGTCC
TCCTTCTCTTCATCCTAGCAACACTAACACTACTCTCACCCAACCTCCTAAACGACCCAGACAACTACAT
TCCAGCCGACCCATTAAACACTCCCCCACATATCAAACCAGAGTGATACTTCCTATTTGCATACACAATC
CTACGATCCATCCCCAACAAACTGGGAGGCGTACTAGCACTCTTTCTATCGATCCTCATTCTAGCAGCCA
TCCCTATACTTCACAAATCCAAACAACAAAGCATAATATTCCGCCCACTCAGCCAATTTCTATTCTGACT
CCTAATCACAATCCTATTGACCCTTACCTGAATTGGAAGCGAACCAGTAGTCCAACCCCTTACCACTATC
GGCCAAGTAGCATCCATAATATACTTCATCACAATTCTAATCCTAATACCACTGGCCTCCCTAATCGAAA
ACAACCTACTCAAGTGAACTT


Overwriting alineamiento.fasta


Vamos a definir el intervalo de tiempo $\Delta t = 1\text{ Ma}$ (1.000.000 años). Así, un tiempo de divergencia de $t$ millones de años corresponde a $n = t / \Delta t$ pasos enteros (por ejemplo, $6\text{ Ma} \to n = 6$ pasos), y las probabilidades de mutación por paso se mantienen pequeñas y bien condicionadas al contexto trabajado.

El tiempo de divergencia de dos especies actuales $X$ e $Y$ que se separaron hace $n$ pasos acumularon $n$ pasos evolutivos según nuestro modelo, pero de cada rama, lo que significa que la probabilidad de observar el nuclótido $j$ en la especie $Y$ dado que la especie $X$ tiene en el lugar equivalente el nuclétido $i$ es lo equivalente a pensar en la evolución de una sola cadena por $m=2n$ pasos.

$$\mathbb{P}(Y_n = j \mid X_n = i) = (P^{2n})_{i,j}$$

Primero vamos a calcular las frecuencias de cambio para cada una de las posiciones para cada una de las especies comparadas con el humano.


$A \to A$ <br>
$A \to T$ <br>
$A \to C$ <br>
$A \to G$ <br> <br>

$T \to A$ <br>
$T \to T$ <br>
$T \to C$ <br>
$T \to G$ <br> <br>

$C \to A$ <br>
$C \to T$ <br>
$C \to C$ <br>
$C \to G$ <br> <br>

$G \to A$ <br>
$G \to T$ <br>
$G \to C$ <br>
$G \to G$ <br>

Para ello, vamos a cargar el multifasta a diccionario.


In [ ]:
def leer_multifasta(ruta):
    seqs = {}
    nombre = None
    with open(ruta, "r") as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            if linea.startswith(">"):
                nombre = linea[1:].split()[0]
                seqs[nombre] = []
            else:
                seqs[nombre].append(linea)
    seqs = {k: "".join(v) for k, v in seqs.items()}
    # Validar que todas midan lo mismo y solo contengan A, G, C, T
    longitudes = {len(s) for s in seqs.values()}
    assert len(longitudes) == 1, f"Longitudes distintas detectadas: {longitudes}"
    for k, s in seqs.items():
        assert set(s).issubset({"A", "G", "C", "T"}), f"Bases inválidas en {k}"
    return seqs

secuencias = leer_multifasta("alineamiento.fasta")
print("Especies cargadas:", list(secuencias.keys()))

L = len(next(iter(secuencias.values())))
print(f"Longitud de las secuencias: {L} pb")

Especies cargadas: ['Homo_sapies', 'Pan_troglodytes', 'Gorilla_gorilla', 'Pongo_abelii', 'Nomascus_leucogenys', 'Macaca_mulatta']
Longitud de las secuencias: 1141 pb


In [ ]:
import numpy as np
import pandas as pd

BASES = ["A", "T", "C", "G"]
IDX = {base: i for i, base in enumerate(BASES)}

def contar_cambios(seq_ref, seq_sp):
    """Devuelve una matriz 4x4 de enteros con los conteos ref -> especie."""
    matriz = np.zeros((4, 4), dtype=int)
    for b_ref, b_sp in zip(seq_ref, seq_sp):
        matriz[IDX[b_ref], IDX[b_sp]] += 1
    return matriz

ref = "Homo_sapies"
conteos_por_especie = {}
prob_por_especie = {}

for especie, seq in secuencias.items():
    if especie == ref:
        continue

    # 1. Matriz de conteos enteros (suman L = 1141 en total)
    M_conteos = contar_cambios(secuencias[ref], seq)
    conteos_por_especie[especie] = M_conteos

    # 2. Matriz de probabilidades condicionales (cada fila suma 1)
    totales_por_fila = M_conteos.sum(axis=1, keepdims=True)
    M_prob = M_conteos / totales_por_fila
    prob_por_especie[especie] = M_prob

# Mostrar resultados ordenados para inspeccionar la evidencia
for especie in conteos_por_especie:
    print(f"\n================ {ref}  --->  {especie} ================")
    df_c = pd.DataFrame(conteos_por_especie[especie], index=BASES, columns=BASES)
    df_p = pd.DataFrame(prob_por_especie[especie], index=BASES, columns=BASES).round(4)

    print("Conteos enteros (total =", conteos_por_especie[especie].sum(), "pb):")
    display(df_c)
    print("Probabilidades condicionales empíricas (cada fila suma 1):")
    display(df_p)


================ Homo_sapies  --->  Pan_troglodytes ================
Conteos enteros (total = 1141 pb):


,A,T,C,G
A,307,0,2,17
T,2,241,44,0
C,2,45,344,0
G,19,0,1,117


Probabilidades condicionales empíricas (cada fila suma 1):


,A,T,C,G
A,0.9417,0.0000,0.0061,0.0521
T,0.0070,0.8397,0.1533,0.0000
C,0.0051,0.1151,0.8798,0.0000
G,0.1387,0.0000,0.0073,0.8540



================ Homo_sapies  --->  Gorilla_gorilla ================
Conteos enteros (total = 1141 pb):


,A,T,C,G
A,304,4,5,13
T,3,238,46,0
C,5,44,341,1
G,20,0,1,116


... (10 outputs left out)


================ Homo_sapies  --->  Macaca_mulatta ================
Conteos enteros (total = 1141 pb):


,A,T,C,G
A,271,11,26,18
T,7,220,56,4
C,25,53,305,8
G,29,3,1,104


Probabilidades condicionales empíricas (cada fila suma 1):


,A,T,C,G
A,0.8313,0.0337,0.0798,0.0552
T,0.0244,0.7666,0.1951,0.0139
C,0.0639,0.1355,0.7801,0.0205
G,0.2117,0.0219,0.0073,0.7591


In [ ]:
from scipy.linalg import logm, expm
import numpy as np
import pandas as pd

TIEMPOS_REALES = {
    "Pan_troglodytes":     6.5,
    "Gorilla_gorilla":     8.5,
    "Pongo_abelii":        14.0,
    "Nomascus_leucogenys": 18.5,
    "Macaca_mulatta":      27.0,
}

DT = 0.1  # 1 paso = 0.1 Ma
P_un_paso_por_especie = {}

for especie, M_conteos in conteos_por_especie.items():
    t_ma = TIEMPOS_REALES[especie]
    m_pasos = int(round(2 * t_ma / DT))  # m = 2n pasos de evolución en total

    # 1. Suavizado de Laplace (+1) y cálculo de probabilidades
    M_suav = M_conteos + 1
    M_prob = M_suav / M_suav.sum(axis=1, keepdims=True)

    # 2. Estimación de la matriz en un paso: P = M_prob ^ (1/m)
    P_raw = expm(logm(M_prob) / m_pasos).real
    P_limpia = np.clip(P_raw, 0.0, None)
    P_limpia = P_limpia / P_limpia.sum(axis=1, keepdims=True)

    P_un_paso_por_especie[especie] = P_limpia

    print(f"\n--- Matriz de transición estimada para {especie} (t = {t_ma} Ma) ---")
    display(pd.DataFrame(P_limpia, index=BASES, columns=BASES).round(6))

# 3. Matriz de transición consenso (promedio simple)
P_consenso = np.mean(list(P_un_paso_por_especie.values()), axis=0)
P_consenso = P_consenso / P_consenso.sum(axis=1, keepdims=True)

print("\n============================================================")
print("   MATRIZ DE TRANSICIÓN CONSENSO P (1 paso = 0.1 Ma)")
print("============================================================")
display(pd.DataFrame(P_consenso, index=BASES, columns=BASES).round(6))


--- Matriz de transición estimada para Pan_troglodytes (t = 6.5 Ma) ---


,A,T,C,G
A,0.999432,0.000020,0.000072,0.000475
T,0.000083,0.998485,0.001405,0.000027
C,0.000058,0.001058,0.998865,0.000019
G,0.001236,0.000056,0.000117,0.998592



--- Matriz de transición estimada para Gorilla_gorilla (t = 8.5 Ma) ---


,A,T,C,G
A,0.999512,0.000094,0.000109,0.000285
T,0.000082,0.998764,0.001134,0.000019
C,0.000092,0.000800,0.999077,0.000032
G,0.001001,0.000036,0.000085,0.998878



--- Matriz de transición estimada para Pongo_abelii (t = 14.0 Ma) ---


,A,T,C,G
A,0.999524,0.000106,0.000152,0.000219
T,0.000082,0.999156,0.000735,0.000027
C,0.000173,0.000474,0.999350,0.000004
G,0.000755,0.000017,0.000043,0.999186


... (2 outputs left out)


--- Matriz de transición estimada para Macaca_mulatta (t = 27.0 Ma) ---


,A,T,C,G
A,0.999616,0.000067,0.000182,0.000134
T,0.000041,0.999446,0.000478,0.000034
C,0.000144,0.000332,0.999477,0.000047
G,0.000505,0.000059,0.000001,0.999434



   MATRIZ DE TRANSICIÓN CONSENSO P (1 paso = 0.1 Ma)


,A,T,C,G
A,0.999533,0.000061,0.000132,0.000274
T,0.000079,0.999023,0.000871,0.000027
C,0.000126,0.000624,0.999224,0.000027
G,0.000771,0.000042,0.000056,0.999132


## Referencias.

Ababneh, F., Jermiin, L. S., Ma, C., & Robinson, J. (2006). Matched-pairs tests of homogeneity with applications to homologous nucleotide sequences. *Bioinformatics*, *22*(10), 1225–1231. https://doi.org/10.1093/bioinformatics/btl064

Barry, D., & Hartigan, J. A. (1987). Asynchronous distance between homologous DNA sequences. *Biometrics*, *43*(2), 261–276.

Bowker, A. H. (1948). A test for symmetry in contingency tables. *Journal of the American Statistical Association*, *43*(244), 572–574.

Brown, W. M., George, M., Jr., & Wilson, A. C. (1979). Rapid evolution of animal mitochondrial DNA. *Proceedings of the National Academy of Sciences of the USA*, *76*(4), 1967–1971. https://doi.org/10.1073/pnas.76.4.1967

Brown, W. M., Prager, E. M., Wang, A., & Wilson, A. C. (1982). Mitochondrial DNA sequences of primates: Tempo and mode of evolution. *Journal of Molecular Evolution*, *18*(4), 225–239. https://doi.org/10.1007/BF01734101

Dayhoff, M. O., Schwartz, R. M., & Orcutt, B. C. (1978). A model of evolutionary change in proteins. En M. O. Dayhoff (Ed.), *Atlas of Protein Sequence and Structure* (Vol. 5, Supl. 3, pp. 345–352). National Biomedical Research Foundation.

Duchêne, S., Ho, S. Y. W., & Holmes, E. C. (2015). Declining transition/transversion ratios through time reveal limitations to the accuracy of nucleotide substitution models. *BMC Evolutionary Biology*, *15*, 36. https://doi.org/10.1186/s12862-015-0312-6

Felsenstein, J. (1981). Evolutionary trees from DNA sequences: A maximum likelihood approach. *Journal of Molecular Evolution*, *17*(6), 368–376. https://doi.org/10.1007/BF01734359

Häggström, O. (2002). *Finite Markov chains and algorithmic applications* (London Mathematical Society Student Texts, Vol. 52). Cambridge University Press.

Hasegawa, M., Kishino, H., & Yano, T. (1985). Dating of the human–ape splitting by a molecular clock of mitochondrial DNA. *Journal of Molecular Evolution*, *22*(2), 160–174. https://doi.org/10.1007/BF02101694

Henikoff, S., & Henikoff, J. G. (1992). Amino acid substitution matrices from protein blocks. *Proceedings of the National Academy of Sciences of the USA*, *89*(22), 10915–10919. https://doi.org/10.1073/pnas.89.22.10915

Ho, S. Y. W., Phillips, M. J., Cooper, A., & Drummond, A. J. (2005). Time dependency of molecular rate estimates and systematic overestimation of recent divergence times. *Molecular Biology and Evolution*, *22*(7), 1561–1568. https://doi.org/10.1093/molbev/msi145

Irwin, D. M., Kocher, T. D., & Wilson, A. C. (1991). Evolution of the cytochrome *b* gene of mammals. *Journal of Molecular Evolution*, *32*(2), 128–144. https://doi.org/10.1007/BF02515385

Jukes, T. H., & Cantor, C. R. (1969). Evolution of protein molecules. En H. N. Munro (Ed.), *Mammalian protein metabolism* (Vol. 3, pp. 21–132). Academic Press.

Kimura, M. (1968). Evolutionary rate at the molecular level. *Nature*, *217*(5129), 624–626. https://doi.org/10.1038/217624a0

Kimura, M. (1980). A simple method for estimating evolutionary rates of base substitutions through comparative studies of nucleotide sequences. *Journal of Molecular Evolution*, *16*(2), 111–120. https://doi.org/10.1007/BF01731581

Kingman, J. F. C. (1962). The imbedding problem for finite Markov chains. *Zeitschrift für Wahrscheinlichkeitstheorie und Verwandte Gebiete*, *1*(1), 14–24. https://doi.org/10.1007/BF00531768

Kumar, S. (2005). Molecular clocks: Four decades of evolution. *Nature Reviews Genetics*, *6*(8), 654–662. https://doi.org/10.1038/nrg1659

Kumar, S., Suleski, M., Craig, J. M., Kasprowicz, A. E., Sanderford, M., Li, M., Stecher, G., & Hedges, S. B. (2022). TimeTree 5: An expanded resource for species divergence times. *Molecular Biology and Evolution*, *39*(8), msac174. https://doi.org/10.1093/molbev/msac174

Lockhart, P. J., Steel, M. A., Hendy, M. D., & Penny, D. (1994). Recovering evolutionary trees under a more realistic model of sequence evolution. *Molecular Biology and Evolution*, *11*(4), 605–612. https://doi.org/10.1093/oxfordjournals.molbev.a040136

Rodríguez, F., Oliver, J. L., Marín, A., & Medina, J. R. (1990). The general stochastic model of nucleotide substitution. *Journal of Theoretical Biology*, *142*(4), 485–501. https://doi.org/10.1016/S0022-5193(05)80104-3

Siepel, A., & Haussler, D. (2004). Phylogenetic estimation of context-dependent substitution rates by maximum likelihood. *Molecular Biology and Evolution*, *21*(3), 468–488.

Squartini, F., & Arndt, P. F. (2008). Quantifying the stationarity and time reversibility of the nucleotide substitution process. *Molecular Biology and Evolution*, *25*(12), 2525–2535. https://doi.org/10.1093/molbev/msn169

Tavaré, S. (1986). Some probabilistic and statistical problems in the analysis of DNA sequences. *Lectures on Mathematics in the Life Sciences*, *17*, 57–86. American Mathematical Society.

Thorne, J. L., Kishino, H., & Painter, I. S. (1998). Estimating the rate of evolution of the rate of molecular evolution. *Molecular Biology and Evolution*, *15*(12), 1647–1657. https://doi.org/10.1093/oxfordjournals.molbev.a025892

Tuffley, C., & Steel, M. (1998). Modeling the covarion hypothesis of nucleotide substitution. *Mathematical Biosciences*, *147*(1), 63–91. https://doi.org/10.1016/S0025-5564(97)00081-3

Verbyla, K. L., Yap, V. B., Pahwa, A., Shao, Y., & Huttley, G. A. (2013). The embedding problem for Markov models of nucleotide substitution. *PLoS ONE*, *8*(7), e69187. https://doi.org/10.1371/journal.pone.0069187

Yang, Z. (1994). Maximum likelihood phylogenetic estimation from DNA sequences with variable rates over sites: Approximate methods. *Journal of Molecular Evolution*, *39*(3), 306–314. https://doi.org/10.1007/BF00160154

Yang, Z. (2014). *Molecular evolution: A statistical approach*. Oxford University Press.

Yang, Z., & Rannala, B. (1997). Bayesian phylogenetic inference using DNA sequences: A Markov chain Monte Carlo method. *Molecular Biology and Evolution*, *14*(7), 717–724. https://doi.org/10.1093/oxfordjournals.molbev.a025811

Zuckerkandl, E., & Pauling, L. (1965). Evolutionary divergence and convergence in proteins. En V. Bryson & H. J. Vogel (Eds.), *Evolving genes and proteins* (pp. 97–166). Academic Press.